In [1]:
import pandas as pd
import numpy as np
import os
from config_paths import USE_TEST_DATA, DATA_FOLDER
from config_variables import VARIABLE_MAP

# --- PATH CONFIGURATION ---
WAVES = ["o"] if USE_TEST_DATA else ["o", "n", "m", "l", "k", "j", "a"]
RAW_DIR    = f"../{DATA_FOLDER}/0_raw/ukhls"
PICKLE_DIR = f"../{DATA_FOLDER}/2_pickle_ukhls_waves"

def run_standardized_ingestion():
    if not os.path.exists(PICKLE_DIR):
        os.makedirs(PICKLE_DIR)

    for w in WAVES:
        ind_path = os.path.join(RAW_DIR, f"{w}_indresp.tab")
        hh_path = os.path.join(RAW_DIR, f"{w}_hhresp.tab")

        if not os.path.exists(ind_path):
            print(f"Skipping Wave {w}: Individual response file not found.")
            continue

        print(f"--- Ingesting Wave {w} ---")

        # 1. SCAN HEADERS
        ind_headers = pd.read_csv(ind_path, sep='\t', nrows=0).columns.tolist()
        hh_exists = os.path.exists(hh_path)
        hh_headers = pd.read_csv(hh_path, sep='\t', nrows=0).columns.tolist() if hh_exists else []

        # 2. SELECT RELEVANT COLUMNS BASED ON VARIABLE_MAP
        ind_to_load = ['pidp']
        hidp_col_name = next((c for c in [f"{w}_hidp", 'hidp'] if c in ind_headers), None)
        if hidp_col_name:
            ind_to_load.append(hidp_col_name)

        hh_to_load = []
        if hh_exists:
            hh_hidp = next((c for c in [f"{w}_hidp", 'hidp'] if c in hh_headers), None)
            if hh_hidp:
                hh_to_load.append(hh_hidp)

        for base in VARIABLE_MAP.keys():
            if base == 'pidp':
                continue
            prefixed = f"{w}_{base}"
            if prefixed in ind_headers:
                ind_to_load.append(prefixed)
            elif prefixed in hh_headers:
                hh_to_load.append(prefixed)

        # 3. LOAD INDIVIDUAL AND HOUSEHOLD DATA
        df_ind = pd.read_csv(ind_path, sep='\t', usecols=ind_to_load, low_memory=False)

        if hh_to_load and len(hh_to_load) > 1:
            df_hh = pd.read_csv(hh_path, sep='\t', usecols=hh_to_load, low_memory=False)
            df = pd.merge(df_ind, df_hh, on=hidp_col_name, how='left')
            print(f"   -> Merged {len(hh_to_load) - 1} household variables.")
        else:
            df = df_ind

        # 4. SAVE PICKLE
        out_file = os.path.join(PICKLE_DIR, f"{w}_indresp_optimized.pkl")
        df.to_pickle(out_file, protocol=5)
        print(f"   -> Saved {len(df.columns)} variables for {len(df):,} rows to {out_file}\n")

if __name__ == "__main__":
    run_standardized_ingestion()


--- Ingesting Wave o ---
   -> Saved 20 variables for 32,849 rows to ../data/2_pickle_ukhls_waves/o_indresp_optimized.pkl

--- Ingesting Wave n ---
   -> Saved 19 variables for 35,471 rows to ../data/2_pickle_ukhls_waves/n_indresp_optimized.pkl

--- Ingesting Wave m ---
   -> Saved 18 variables for 27,998 rows to ../data/2_pickle_ukhls_waves/m_indresp_optimized.pkl

--- Ingesting Wave l ---
   -> Saved 22 variables for 29,271 rows to ../data/2_pickle_ukhls_waves/l_indresp_optimized.pkl

--- Ingesting Wave k ---
   -> Saved 18 variables for 32,008 rows to ../data/2_pickle_ukhls_waves/k_indresp_optimized.pkl

--- Ingesting Wave j ---
   -> Saved 21 variables for 34,319 rows to ../data/2_pickle_ukhls_waves/j_indresp_optimized.pkl

--- Ingesting Wave a ---
   -> Saved 21 variables for 50,994 rows to ../data/2_pickle_ukhls_waves/a_indresp_optimized.pkl

